# C5.01 Corpus agenti - Tema 2


Scop: ne uităm la `data/corpus_typed.json`, verificăm distribuția bulelor și alegem ce exemple merită păstrate pentru vector store.



## 1. Setup

In [1]:
from pathlib import Path
import os
import pandas as pd

PROJECT_ROOT = Path(r"C:\Users\diana\Desktop\18.Ingineria_AI\echochamber-project-team3")
os.chdir(PROJECT_ROOT)
DATA_PATH = Path("data/typed/corpus_typed.jsonl")


In [2]:
DATA_PATH

WindowsPath('data/typed/corpus_typed.jsonl')

## 2. Încărcăm corpusul

In [3]:
df = pd.read_json(DATA_PATH, lines=True)
df.head(2)

,id,source_channel,channel_family,video_title,text,low_information,pre_filtered,target_specific,target_refined,target_l1,...,repr_pluralist_present,repr_pluralist_strength,dem_procedure_rejected,dem_procedure_defended,call_to_action_present,call_to_action_strength,confidence,discourse_type,discourse_subtype,type_confidence
0,yt_iH8jB4NlV9Y_UgxGvCWbzsvB3I-nSy54AaABAg,georgesimionoficial,sovereigntist,Episodul 2: Cum ne-au furat alegerile - Turism...,Metoda votului cetatenilor din Republica Moldo...,False,False,nicusor_dan,nicusor_dan,Instituții stat,...,False,0,False,False,True,1,0.9,T2_grievance_anti_sistem,grievance_conspiratorial,high
1,yt_iH8jB4NlV9Y_UgwiYbOpe89FfXbeIpl4AaABAg,georgesimionoficial,sovereigntist,Episodul 2: Cum ne-au furat alegerile - Turism...,Doamne cu atâtea dovezi și nicio instituție in...,False,False,simion,simion,Sovereigntist,...,False,0,False,False,False,0,0.9,T1_suport_personalist,suport_afectiv_suveranist,medium


## 3. Vedem structura datelor

In [4]:
len(df)

17886

In [5]:
print("Coloane:")
df.columns

Coloane:


Index(['id', 'source_channel', 'channel_family', 'video_title', 'text',
       'low_information', 'pre_filtered', 'target_specific', 'target_refined',
       'target_l1', 'target_l2', 'stance_to_target', 'primary_target_hint',
       'target_confidence', 'inst_neg_present', 'inst_neg_strength',
       'inst_pos_present', 'inst_pos_strength',
       'epist_hidden_coordination_present',
       'epist_hidden_coordination_strength',
       'epist_evidence_verification_present',
       'epist_evidence_verification_strength',
       'geo_anti_external_domination_present',
       'geo_anti_external_domination_strength',
       'geo_pro_external_anchoring_present',
       'geo_pro_external_anchoring_strength', 'repr_personalist_present',
       'repr_personalist_strength', 'repr_pluralist_present',
       'repr_pluralist_strength', 'dem_procedure_rejected',
       'dem_procedure_defended', 'call_to_action_present',
       'call_to_action_strength', 'confidence', 'discourse_type',
       'disco

In [6]:
df["discourse_type"].value_counts(dropna=False)

discourse_type
T2_grievance_anti_sistem      7583
T1_suport_personalist         4624
T6_afectiv_pozitional         3726
T4_conspiratie_externalism    1506
T3_opozitie_suveranista        261
T5_pro_democratic_european     186
Name: count, dtype: int64

In [26]:
## Exemplu de text pentru fiecare tip de discurs

In [7]:
for bubble in df["discourse_type"].value_counts().index:
    print("\n" + "="*80)
    print(bubble)
    print("="*80)
    
    sample = df[df["discourse_type"] == bubble]["text"].dropna().head(5)
    
    for i, text in enumerate(sample, 1):
        print(f"\n{i}. {text[:50]}")


T2_grievance_anti_sistem

1. Metoda votului cetatenilor din Republica Moldova a

2. Calin Georgescu si George Simion la puscarieeee!! 

3. S-au furat prin liste suplimentare, într-un sat di

4. Eu nu cred că caracatița poate fi dată la parte !E

5. Știu ca există un sistem de vot electronic în care

T1_suport_personalist

1. Doamne cu atâtea dovezi și nicio instituție intern

2. Noi oricum știm asta dar întrebarea mea este domnu

3. Felicitări și RESPECT George Simion! Nu te opri! V

4. Haideți in stradă pe 20 iulie!!!! Să dăm jos hoții

5. MILIOANE DE ROMÂNI DEMNI SUNTEM ALĂTURI DE TINE,GE

T6_afectiv_pozitional

1. Adevarul sa fie facut prezent si rezultatul sa fie

2. Ceea ce face este demn de noi romanii adevarati . 

3. Si de trebuie deposite de combustibil degeaba avem

4. Dumnezeu sa Binecuvinteze Partidul Aur, 🇹🇩🙏 Români

5. Felicitari pentru discurs! ❤️🙏🇷🇴 Si pentru consecv

T4_conspiratie_externalism

1. Atunci daca aveți dovezi împotriva la Maya Sandu d

2. Dom-le am văzut 

# 4. Pastram doar 150 de bula

In [8]:
# Alegem un subset simplu pentru curs: maximum 150 texte per bulă

N_PER_BUBBLE = 150

df_sample = (
    df[df["discourse_type"].notna()]
    .sort_values("type_confidence", ascending=False)
    .groupby("discourse_type", group_keys=False)
    .head(N_PER_BUBBLE)
    .copy()
)

df_sample["discourse_type"].value_counts()

discourse_type
T3_opozitie_suveranista       150
T2_grievance_anti_sistem      150
T1_suport_personalist         150
T4_conspiratie_externalism    150
T5_pro_democratic_european    150
T6_afectiv_pozitional         150
Name: count, dtype: int64

In [9]:
# Păstrăm doar coloanele utile și salvăm corpusul pentru etapa următoare

keep_cols = [
    "id", "text", "source_channel", "channel_family", "video_title",
    "target_refined", "stance_to_target",
    "confidence", "discourse_type", "discourse_subtype", "type_confidence"
]

df_sample = df_sample[keep_cols].drop_duplicates(subset="text").copy()

OUT_PATH = Path("data/typed/corpus_c5_sample.jsonl")
df_sample.to_json(OUT_PATH, orient="records", lines=True, force_ascii=False)

print(df_sample.shape)
print(OUT_PATH)
df_sample.head(2)

(896, 11)
data\typed\corpus_c5_sample.jsonl


,id,text,source_channel,channel_family,video_title,target_refined,stance_to_target,confidence,discourse_type,discourse_subtype,type_confidence
17885,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Am toată încrederea că oameni ( de bine ) ca :...,AlephNewsOfficial,mainstream,ATENȚIE: România e „binevenită” să aplice iar ...,simion,anti,0.9,T3_opozitie_suveranista,opozitie_difuza,medium
15394,yt_joXkZDqGZQU_Ugyqb1XZ7P8GTnJS_4p4AaABAg,Semneaza Bo$$ ca la urmatoarele alegerii nu ma...,NicusorDanRO,mainstream_actor,🟢 Declarații de presă comune cu Președintele U...,nicusor_dan,anti,0.9,T2_grievance_anti_sistem,grievance_mobilizator,medium


## 5. Exportul bulelor finale
În această etapă transformăm corpusul tipologizat în fișiere separate, câte unul pentru fiecare bulă finală.
Codurile `T1`, `T2`, `T3`, `T4`, `T5` vin din adnotare, dar în aplicație folosim numele finale ale agenților:
| Agent | Personalitate | Cum vorbește | Ce îl definește |
|---|---|---|---|
| Personalist-salvator | devotat, admirativ, sigur | laudativ, emoțional, încrezător | vede liderul ca soluție excepțională |
| Anti-sistem | furios, suspicios, dezamăgit | acuzator, moralizator, direct | vede instituțiile și „sistemul” ca profund compromise |
| Anti-suveranist | critic, vigilent, defensiv | contestatar, mai argumentativ | respinge liderii și discursul suveranist |
| Conspiraționist | alarmist, hiper-suspicios | speculativ, revelator, totalizant | explică evenimentele prin forțe ascunse și actori externi |
| Pro-european | normativ, moderat, legalist | sobru, justificativ, procedural | apără regulile, instituțiile și ancorarea europeană |
Intelectual-critic | analitic, sceptic, detasat | calculat, elocvent | analizeaza orice argument si pune in balanta totul inainte de a lua o decizie|

In [10]:
BUBBLES = {
    "T1_suport_personalist": {
        "agent": "Personalist-salvator",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "T2_grievance_anti_sistem": {
        "agent": "Anti-sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "T3_opozitie_suveranista": {
        "agent": "Anti-suveranist",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "T4_conspiratie_externalism": {
        "agent": "Conspiraționist",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "T5_pro_democratic_european": {
        "agent": "Pro-european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
    "T6_afectiv_pozitional": {
        "agent": "Intelectual-critic",
        "slug": "intelectual_critic",
        "personality": "analitic, sceptic, detașat",
        "speaks": "calculat, elocvent",
        "definition": "analizează orice argument și pune în balanță totul înainte de a lua o decizie",
    },
}

In [11]:
"""
OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_PER_BUBBLE = 50

for old_type, meta in BUBBLES.items():
    bubble_df = (
        df_sample[df_sample["discourse_type"] == old_type]
        .drop_duplicates(subset="text")
        .head(N_PER_BUBBLE)
        .copy()
    )

    bubble_df["agent"] = meta["agent"]
    bubble_df["slug"] = meta["slug"]
    bubble_df["personality"] = meta["personality"]
    bubble_df["speaks"] = meta["speaks"]
    bubble_df["definition"] = meta["definition"]
    bubble_df["source_type"] = old_type

    out_path = OUT_DIR / f"{meta['slug']}.jsonl"
    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

    print(f"{meta['agent']}: {len(bubble_df)} texte -> {out_path}")

"""

'\nOUT_DIR = Path("data/bubbles")\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nN_PER_BUBBLE = 50\n\nfor old_type, meta in BUBBLES.items():\n    bubble_df = (\n        df_sample[df_sample["discourse_type"] == old_type]\n        .drop_duplicates(subset="text")\n        .head(N_PER_BUBBLE)\n        .copy()\n    )\n\n    bubble_df["agent"] = meta["agent"]\n    bubble_df["slug"] = meta["slug"]\n    bubble_df["personality"] = meta["personality"]\n    bubble_df["speaks"] = meta["speaks"]\n    bubble_df["definition"] = meta["definition"]\n    bubble_df["source_type"] = old_type\n\n    out_path = OUT_DIR / f"{meta[\'slug\']}.jsonl"\n    bubble_df.to_json(out_path, orient="records", lines=True, force_ascii=False)\n\n    print(f"{meta[\'agent\']}: {len(bubble_df)} texte -> {out_path}")\n\n'

## 6. Alege agentul tău și verifică textele
Fiecare membru al echipei lucrează pe un singur agent.
Datele vin din `df_sample`, iar agentul este legat de eticheta tehnică `discourse_type`.
Scopul este să alegi aproximativ 50 texte bune pentru agentul tău.

In [12]:
# Alegeți un agent și încărcați textele corespunzătoare pentru etapa următoare
AGENTS = {
    "Personalist-salvator": {
        "type": "T1_suport_personalist",
        "slug": "personalist_salvator",
        "personality": "devotat, admirativ, sigur",
        "speaks": "laudativ, emoțional, încrezător",
        "definition": "vede liderul ca soluție excepțională",
    },
    "Anti-sistem": {
        "type": "T2_grievance_anti_sistem",
        "slug": "anti_sistem",
        "personality": "furios, suspicios, dezamăgit",
        "speaks": "acuzator, moralizator, direct",
        "definition": "vede instituțiile și „sistemul” ca profund compromise",
    },
    "Anti-suveranist": {
        "type": "T3_opozitie_suveranista",
        "slug": "anti_suveranist",
        "personality": "critic, vigilent, defensiv",
        "speaks": "contestatar, mai argumentativ",
        "definition": "respinge liderii și discursul suveranist",
    },
    "Conspiraționist": {
        "type": "T4_conspiratie_externalism",
        "slug": "conspirationist",
        "personality": "alarmist, hiper-suspicios",
        "speaks": "speculativ, revelator, totalizant",
        "definition": "explică evenimentele prin forțe ascunse și actori externi",
    },
    "Pro-european": {
        "type": "T5_pro_democratic_european",
        "slug": "pro_european",
        "personality": "normativ, moderat, legalist",
        "speaks": "sobru, justificativ, procedural",
        "definition": "apără regulile, instituțiile și ancorarea europeană",
    },
    "Intelectual-critic": {
        "type": "T6_afectiv_pozitional",
        "slug": "intelectual_critic",
        "personality": "analitic, sceptic, detașat",
        "speaks": "calculat, elocvent",
        "definition": "analizează orice argument și pune în balanță totul înainte de a lua o decizie",
    },
}

MY_AGENT = "Anti-suveranist"  # Alegeți unul dintre agenți: "Personalist-salvator", "Anti-sistem", "Anti-suveranist", "Conspiraționist", "Pro-european"

meta = AGENTS[MY_AGENT]

my_df = (
    df_sample[df_sample["discourse_type"] == meta["type"]]
    .drop_duplicates(subset="text")
    .copy()
)

print("Agent:", MY_AGENT)
print("Tip discurs:", meta["type"])
print("Texte disponibile:", len(my_df))

my_df[["id", "type_confidence", "discourse_subtype", "text"]].head(2)

Agent: Anti-suveranist
Tip discurs: T3_opozitie_suveranista
Texte disponibile: 150


,id,type_confidence,discourse_subtype,text
17885,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,medium,opozitie_difuza,Am toată încrederea că oameni ( de bine ) ca :...
15410,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,medium,opozitie_difuza,Apropo de avalansa de troli ce se devarsa si a...


### Cum verifici textele
Citește textele afișate mai jos.
Dacă un text este slab, copiază ID-ul lui în lista `REMOVE_IDS`.
Elimină texte prea scurte, duplicate, ambigue sau care nu exprimă clar vocea agentului.|

- pot sa folosesc un notepad pentru IDs
daca nu gasesti 50 de comentarii bune, incarca mai multe.
- poti folosi si alte metode (export text, csv) pentru vizualizarea si alegerea comentariilor

In [13]:
for _, row in my_df.head(70).iterrows():
    print("=" * 80)
    print("ID:", row["id"])
    print("Confidence:", row["type_confidence"])
    print("Subtype:", row["discourse_subtype"])
    print(row["text"][:700])

ID: yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg
Confidence: medium
Subtype: opozitie_difuza
Am toată încrederea că oameni ( de bine ) ca : G Simion , Călinge , dna Găurilă ...... vor avea mare grijă să pună fie piedici , fie bețe-n roate astfel încât să rămanem sub tutela cremlinului
ID: yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg
Confidence: medium
Subtype: opozitie_difuza
Apropo de avalansa de troli ce se devarsa si acum aici si fac apologia nebuniei prin invective, amenintari si minciuni. De cel putin 10 ani blochez si raportez la youtube zilnic uneori zeci de troli/boti cu narative putiniste ce probabil majoritatea vin din softul de boti fara numar, ce fac atacurile astea cibernetice. *Cam cu doua zile inainte de primul tur al alegerilor, au inceput sa se deverse in rafale, repetat, prin mai toate subiectele de pe Youtube, comentarii si indemnuri venite de la diverse conturi de troli sau boti, de a-l vota pe calin georgescu. Nici nu stiam cine mai e si asta. Era socant cum chiar in ziu

In [17]:
# elimin textele slabe și păstrez 50

REMOVE_IDS = [
    "yt_6hgc90sLVFw_UgzKEAwJ4lrjMa0IzeB4AaABAg",
    "yt_R-wmsuFxku4_UgyZ1UUIKqIDPM-tch94AaABAg",
    "yt_5QYDDejaR5E_UgzI_aUlMpDiNpab01h4AaABAg"
]

clean_df = (
    my_df[~my_df["id"].isin(REMOVE_IDS)]
    .head(50)
    .copy()
)

clean_df["agent"] = MY_AGENT
clean_df["slug"] = meta["slug"]
clean_df["personality"] = meta["personality"]
clean_df["speaks"] = meta["speaks"]
clean_df["definition"] = meta["definition"]

print("Texte finale:", len(clean_df))
clean_df[["id", "agent", "text"]].head(10)

Texte finale: 50


,id,agent,text
17885,yt_Tx8GhU2LeyI_UgwoWOyzF2UbPYnguUB4AaABAg,Anti-suveranist,Am toată încrederea că oameni ( de bine ) ca :...
15410,yt_6_Hc2S02Duw_Ugytw6-BDQ2pA_Zi-TB4AaABAg,Anti-suveranist,Apropo de avalansa de troli ce se devarsa si a...
3276,yt_bee6nXyzJ_E_UgxQ1N1kdP_MTx8B4K14AaABAg,Anti-suveranist,Aceasta nu este o emisiune....este o regizare ...
3277,yt_bee6nXyzJ_E_Ugyjnx0utsCXEXc94q54AaABAg,Anti-suveranist,La pregatit bine Putin a investit bani in Guru...
3215,yt_im3QoqSgfDo_UgwOL2VI3_toiLZSDqh4AaABAg,Anti-suveranist,"Eu am vorbit cu susținători de ai lui CG, îs d..."
2882,yt_OP3PA47JM_M_UgwrxXRV2SQUdDLdbFd4AaABAg,Anti-suveranist,Grindeanu este o Matriosca la fel și Simion. D...
15185,yt_N5Wr2OHINDo_Ugw9Gyk4Ehe3UofP07V4AaABAg,Anti-suveranist,Adică UDMR a fost cu Psd tot timpul la guverna...
15206,yt_DzqO3anbtUE_UgzcIZK1ev4kPy8en6V4AaABAg,Anti-suveranist,Cu ce te încălzește ca stai pe o mină de aur d...
15066,yt_pfOUEvkf38M_UgydpTbStbf2J2sC1Bp4AaABAg,Anti-suveranist,Mulțumim pentru explicații! 👍🏼 Acest GS este u...
3472,yt_bee6nXyzJ_E_UgwMwq1Jvcel1wkruXl4AaABAg,Anti-suveranist,A Alexandrescu. Ești super deșteaptă felicitar...


### Descrierea agentului tău
Înainte să exporți bula curată, descrie în 5–7 rânduri ce fel de voce discursivă are agentul ales.
Nu descrie o persoană reală. Descrie un tip de discurs observat în corpus.
Răspunde la aceste întrebări:
- Cum vede acest agent instituțiile, politica sau actorii publici?
- Ce ton folosește cel mai des?
- Ce tip de argumente sau acuzații apar frecvent?
- Ce îl diferențiază de celelalte bule?
- Ce ar trebui să păstreze un viitor agent AI ca să sune coerent cu această bulă?

descriere:
- Acest agent apără democrația, pluralismul, orientarea pro-occidentală a țării, vede curentul naționalist-populist și liderii suveraniști nu ca pe niște salvatori, ci ca pe niște actori periculoși
- Tonul este unul contestatar, mai argumentativ, fiind caracterizat si de putin scepticism fata de celelalte inclinari politice, folosind adesea ironii
- Acuza liderii suveranisti de manipuari, de utiizarea botilor, propaganda de minciuni, utilizarea aberanta a motivelor religioase
- Analizeaza mai la rece lucrurile pe care le condamna, incercand sa aduca argumente logice in loc de unele personale
- Un agent AI ar trebui sa invete sa formuleze propozitii ironice si sarcastice, sa ceara argumente, sa nu cada in plasele religioase si sa taxeze dezinformarile, minciunile si omiterea adevarului complet

In [18]:
# export bula curată pentru etapa următoare

OUT_DIR = Path("data/bubbles")
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"{meta['slug']}.jsonl"

clean_df.to_json(out_path, orient="records", lines=True, force_ascii=False)

print("Salvat:", out_path)

Salvat: data\bubbles\anti_suveranist.jsonl
